<a href="https://colab.research.google.com/github/giabq/plnEp2/blob/main/modelo_final/modeloFinal_com_rotulacao_teste2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rotulação do conjunto de teste 2 com modelo final
Ordem de execução

- Célula 3. Confira que imprime Device: cuda.


- Célula 6 (a de clonar o repositório). Ela deve terminar com Dados carregados e validados. e mostrar 20092 no treino e 900 no teste 1. Se der erro de arquivo faltando, ela lista quais são.


- Suba a pasta modelo_final para /content/plnEp2/modelo_final (descompacte se for zip) e confira que test2.xlsx e test2_limpo.csv estão em /content/plnEp2/data.


- Célula preparar_arquivos.py, primeira vez, com BERT_SRC = None. Ela copia os .npy e lista as pastas que parecem conter um modelo BERT. Veja se algum .npy aparece como “NÃO ACHEI”.


- Escolha a pasta do BERT Head-Tail original na lista, coloque o caminho em BERT_SRC e rode a célula de novo. No final deve aparecer BERT final presente: True. Se houver mais de uma pasta e você não souber qual é, pergunte à outra pessoa. A célula final confere se é a certa.


- Célula 8. Deve mostrar Resultados encontrados. Carregando... para os dois modelos TF-IDF. Se aparecer INICIANDO sem essa frase, ela está recalculando, o que significa que os arquivos não foram copiados.


- Célula 12. Deve mostrar Embeddings Qwen 0.6B encontrados, Previsões encontradas. Carregando... e Embeddings Qwen 4B encontrados. Se começar a imprimir Carregando: Qwen/... e Processados: .../20092, pare, porque está tentando refazer tudo.


- Célula final nova. Ela leva alguns minutos por causa do Qwen 4B. No fim, confira a concordância com o teste 1 (perto de 1,0), a distribuição dos rótulos e a mensagem ROTULAÇÃO DO TESTE 2 CONCLUÍDA.

#Instalação, imports, configurações e GPU

In [12]:
import os
import gdown

url_pasta_drive = "https://drive.google.com/drive/folders/1pl4HD7_Dops-V30ae6hnimm9-j-aM2yG?usp=sharing"
destino = "/content/plnEp2/modelo_final"

os.makedirs(destino, exist_ok=True)

print("Baixando a pasta modelo_final do Google Drive...")
gdown.download_folder(url_pasta_drive, output=destino, quiet=False)

print("Tudo pronto! Os modelos e probabilidades foram carregados com sucesso.")

Baixando a pasta modelo_final do Google Drive...


Retrieving folder contents


Retrieving folder 1vEmkdf0UQgcZtwQhq__O_K9Cf8H_MBEf embeddings
Processing file 1eUiUG3f34kQW9wkPMshznGQHD317PBcZ qwen4_test1_embeddings.npy
Processing file 16QTnkRS6ye_WsjPJ-qFUpoX_Nfd7rZHF qwen4_train_embeddings.npy
Processing file 1nRYy3vLYeEChOY9y0iVtYlj1zGtEPDac qwen06_test1_embeddings.npy
Processing file 1TYeM5JeeC-x2mObWdex5GOTw-z9Vf0aF qwen06_train_embeddings.npy
Retrieving folder 1SgnhslH2aGlT0BqOlB7HTFAUppfpDiDH metadados
Processing file 1U25S1MOuirPllE-y3KZmuLy785Wokbgy modelo_final_metadata.json
Processing file 1ErjSrAeQ5W3kWCNaNqV4Z8pGoQgl8DQx y_test1_encoded.npy
Processing file 1HtZf5gaZOfEgGfhJ_AIu1zbAK8kWAV20 y_train_encoded.npy
Retrieving folder 1c8607r7tD4Y6gwlIsUfNSOi4P8HqjOdW modelos
Retrieving folder 1lOl_DDdKr5-eGUpjVcEbxXRnQ2dpVw4z bert_headtail_final_model
Processing file 1LyAWT_MKzbb5NDHDdL9OdIP7NC8TFhPw config.json
Processing file 1jRtXiHhSvmpuDOKo-grcX42p2x_tC1Au model.safetensors
Processing file 1tA4JnQESolQIY2g1OId3vjDVsU9ry1ta special_tokens_map.json
Proces

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1eUiUG3f34kQW9wkPMshznGQHD317PBcZ
To: /content/plnEp2/modelo_final/embeddings/qwen4_test1_embeddings.npy
100%|██████████| 9.22M/9.22M [00:00<00:00, 17.8MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=16QTnkRS6ye_WsjPJ-qFUpoX_Nfd7rZHF
From (redirected): https://drive.google.com/uc?id=16QTnkRS6ye_WsjPJ-qFUpoX_Nfd7rZHF&confirm=t&uuid=f7fb6424-994e-40b2-b687-4d991d023d6e
To: /content/plnEp2/modelo_final/embeddings/qwen4_train_embeddings.npy
100%|██████████| 206M/206M [00:05<00:00, 38.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1nRYy3vLYeEChOY9y0iVtYlj1zGtEPDac
To: /content/plnEp2/modelo_final/embeddings/qwen06_test1_embeddings.npy
100%|██████████| 3.69M/3.69M [00:00<00:00, 222MB/s]
Downloading...
From: https://drive.google.com/uc?id=1TYeM5JeeC-x2mObWdex5GOTw-z9Vf0aF
To: /content/plnEp2/modelo_final/e

Tudo pronto! Os modelos e probabilidades foram carregados com sucesso.



Download completed


In [13]:
# @title
# ============================================================
# 1. INSTALAÇÃO
# ============================================================

!pip install -q "transformers>=4.51.0,<5" accelerate sentencepiece safetensors openpyxl

# ============================================================
# 2. IMPORTS
# ============================================================

import os
import gc
import json
import random
import shutil
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

warnings.filterwarnings("ignore")

# ============================================================
# 3. REPRODUTIBILIDADE
# ============================================================

SEED = 42

def set_seed(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

# ============================================================
# 4. CONFIGURAÇÕES GERAIS
# ============================================================

CLASSES = ["c1", "c234", "c5"]

LABEL_TO_ID = {
    label: index
    for index, label in enumerate(CLASSES)
}

ID_TO_LABEL = {
    index: label
    for label, index in LABEL_TO_ID.items()
}

N_SPLITS = 3

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Diretório de resultados
OUTPUT_DIR = "/content/ensemble_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Device:", DEVICE)

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    )

    free_memory = torch.cuda.mem_get_info()[0] / 1024**3

    print(f"VRAM total: {total_memory:.2f} GB")
    print(f"VRAM disponível: {free_memory:.2f} GB")

else:

    print("AVISO: nenhuma GPU detectada.")

print("\nClasses:", CLASSES)
print("Diretório de saída:", OUTPUT_DIR)

Device: cuda
GPU: Tesla T4
VRAM total: 14.56 GB
VRAM disponível: 14.46 GB

Classes: ['c1', 'c234', 'c5']
Diretório de saída: /content/ensemble_outputs


#Repositório, carregamento e validação dos dados

In [14]:
# @title
# ============================================================
# 1. CLONAR / ATUALIZAR REPOSITÓRIO
# ============================================================

import os
import shutil
import subprocess

REPO_URL = "https://github.com/giabq/plnEp2.git"
REPO_DIR = "/content/plnEp2"
TEMP_REPO_DIR = "/content/plnEp2_clone_temp"

if not os.path.exists(REPO_DIR):
    # Caso a pasta não exista, clona diretamente
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

elif os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Caso já seja um repositório Git, atualiza
    print("Repositório Git encontrado. Atualizando...")

    subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "--ff-only"],
        check=True
    )

else:
    # A pasta existe, mas não é um repositório Git.
    # Preserva seu conteúdo e clona em uma pasta temporária.
    print("Pasta existente não é um repositório Git.")
    print("Clonando o repositório em uma pasta temporária...")

    if os.path.exists(TEMP_REPO_DIR):
        shutil.rmtree(TEMP_REPO_DIR)

    subprocess.run(
        ["git", "clone", REPO_URL, TEMP_REPO_DIR],
        check=True
    )

    # Copia os arquivos do clone para a pasta original.
    # Não sobrescreve arquivos já existentes.
    shutil.copytree(
        TEMP_REPO_DIR,
        REPO_DIR,
        dirs_exist_ok=True
    )

    # Remove somente a cópia temporária.
    shutil.rmtree(TEMP_REPO_DIR)

    print("Repositório restaurado sem apagar os arquivos existentes.")

DATA_DIR = os.path.join(REPO_DIR, "data")

# ============================================================
# 2. CAMINHOS
# ============================================================

TRAIN_RAW_PATH = os.path.join(DATA_DIR, "train.csv")
TRAIN_CLEAN_PATH = os.path.join(DATA_DIR, "train_limpo.csv")

TEST1_RAW_PATH = os.path.join(DATA_DIR, "test1.xlsx")
TEST1_CLEAN_PATH = os.path.join(DATA_DIR, "test1_limpo.csv")

TEST1_LABELS_PATH = os.path.join(
    DATA_DIR,
    "test1-rotulado-para-teste-estatistico.xlsx"
)

# Verifica se os arquivos necessários existem
required_files = [
    TRAIN_RAW_PATH,
    TRAIN_CLEAN_PATH,
    TEST1_RAW_PATH,
    TEST1_CLEAN_PATH,
    TEST1_LABELS_PATH
]

missing_files = [
    path for path in required_files
    if not os.path.isfile(path)
]

if missing_files:
    raise FileNotFoundError(
        "Os seguintes arquivos não foram encontrados:\n"
        + "\n".join(missing_files)
        + "\n\nVerifique se eles estão disponíveis no repositório "
          "ou se precisam ser adicionados manualmente."
    )

# ============================================================
# 3. CARREGAMENTO
# ============================================================

train_raw = pd.read_csv(TRAIN_RAW_PATH)
train_clean = pd.read_csv(TRAIN_CLEAN_PATH)

test1_original = pd.read_excel(TEST1_RAW_PATH)
test1_clean = pd.read_csv(TEST1_CLEAN_PATH)
test1_labels = pd.read_excel(TEST1_LABELS_PATH)

# ============================================================
# 4. VERIFICAÇÃO DAS COLUNAS
# ============================================================

required_columns = {"resp_text", "clarity"}

datasets = {
    "train_raw": train_raw,
    "train_clean": train_clean,
    "test1_original": test1_original,
    "test1_clean": test1_clean,
    "test1_labels": test1_labels
}

for name, df in datasets.items():

    missing = required_columns - set(df.columns)

    if missing:

        raise ValueError(
            f"{name}: colunas ausentes: {missing}"
        )

# ============================================================
# 5. VERIFICAÇÃO DE ALINHAMENTO
# ============================================================

assert len(train_raw) == len(train_clean), \
    "Quantidade de linhas diferente no treinamento."

assert len(test1_original) == len(test1_clean), \
    "Quantidade de linhas diferente no Teste 1 limpo."

assert len(test1_original) == len(test1_labels), \
    "Quantidade de linhas diferente entre Teste 1 e arquivo rotulado."

assert train_raw["clarity"].astype(str).equals(
    train_clean["clarity"].astype(str)
), "Os rótulos do treinamento não estão alinhados."

assert test1_original["resp_text"].fillna("").astype(str).equals(
    test1_labels["resp_text"].fillna("").astype(str)
), "Os textos originais do Teste 1 não estão alinhados."

# ============================================================
# 6. NORMALIZAÇÃO MÍNIMA DOS TEXTOS ORIGINAIS
# ============================================================

def normalize_raw_text(series):

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .tolist()
    )


X_train_raw = normalize_raw_text(
    train_raw["resp_text"]
)

X_test1_raw = normalize_raw_text(
    test1_original["resp_text"]
)

X_train_clean = normalize_raw_text(
    train_clean["resp_text"]
)

X_test1_clean = normalize_raw_text(
    test1_clean["resp_text"]
)

# ============================================================
# 7. RÓTULOS
# ============================================================

y_train = train_raw["clarity"].astype(str).values

y_test1 = test1_labels["clarity"].astype(str).values

unexpected_train = set(np.unique(y_train)) - set(CLASSES)
unexpected_test = set(np.unique(y_test1)) - set(CLASSES)

assert not unexpected_train, \
    f"Classes inesperadas no treinamento: {unexpected_train}"

assert not unexpected_test, \
    f"Classes inesperadas no Teste 1: {unexpected_test}"

y_train_encoded = np.array([
    LABEL_TO_ID[label]
    for label in y_train
])

y_test1_encoded = np.array([
    LABEL_TO_ID[label]
    for label in y_test1
])

# ============================================================
# 8. VERIFICAÇÃO FINAL
# ============================================================

print("Treinamento:", len(X_train_raw))
print("Teste 1:", len(X_test1_raw))

print("\nDistribuição do treinamento:")
print(pd.Series(y_train).value_counts())

print("\nDistribuição do Teste 1:")
print(pd.Series(y_test1).value_counts())

print("\nComprimento médio dos textos:")

print(
    "Treinamento original:",
    np.mean([len(x) for x in X_train_raw])
)

print(
    "Treinamento limpo:",
    np.mean([len(x) for x in X_train_clean])
)

print("\nDados carregados e validados.")

Repositório Git encontrado. Atualizando...
Treinamento: 20092
Teste 1: 900

Distribuição do treinamento:
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64

Distribuição do Teste 1:
c5      300
c1      300
c234    300
Name: count, dtype: int64

Comprimento médio dos textos:
Treinamento original: 998.3792056539917
Treinamento limpo: 918.3129603822417

Dados carregados e validados.


In [3]:
!unzip -q /content/plnEp2/modelo_final.zip -d /content/plnEp2/

unzip:  cannot find or open /content/plnEp2/modelo_final.zip, /content/plnEp2/modelo_final.zip.zip or /content/plnEp2/modelo_final.zip.ZIP.


## EU ADICIONEI

In [15]:
# ============================================================
# PREPARAR ARQUIVOS PARA A ROTULAÇÃO DO TESTE 2
# Rode DEPOIS da célula "CLONAR / ATUALIZAR REPOSITÓRIO" (célula 6)
# e ANTES das células 8 e 12.
#
# Copia as probabilidades, embeddings e o BERT salvos pela outra
# pessoa para /content/ensemble_outputs, com os nomes que o
# notebook espera. Não altera nem apaga nada na pasta de origem.
# ============================================================

import os
import shutil

# Pasta onde estão os arquivos recebidos (modelo_final).
# Se você subiu/descompactou em outro lugar, troque este caminho.
SRC = "/content/plnEp2/modelo_final"

DST = "/content/ensemble_outputs"
os.makedirs(DST, exist_ok=True)

if not os.path.isdir(SRC):
    raise FileNotFoundError(
        f"Pasta não encontrada: {SRC}. "
        "Suba a pasta modelo_final (ou o zip) para o Colab e ajuste SRC."
    )

# ------------------------------------------------------------
# 1. Arquivos .npy esperados pelo notebook
# ------------------------------------------------------------

needed = [
    "tfidf_word_oof.npy",
    "tfidf_word_test1.npy",
    "qwen06_lr_oof.npy",
    "qwen06_lr_test1.npy",
    "qwen4_lr_oof.npy",
    "qwen4_lr_test1.npy",
    "qwen06_train_embeddings.npy",
    "qwen06_test1_embeddings.npy",
    "qwen4_train_embeddings.npy",
    "qwen4_test1_embeddings.npy",
    "bert_headtail_oof.npy",
    "bert_headtail_test1.npy",
]

index = {}

for root, dirs, files in os.walk(SRC):
    for filename in files:
        index.setdefault(filename, []).append(
            os.path.join(root, filename)
        )

missing = []

for name in needed:

    paths = index.get(name, [])

    if len(paths) == 1:
        shutil.copy2(paths[0], os.path.join(DST, name))
        print("OK        ", name, "<-", paths[0])

    elif len(paths) > 1:
        print("AMBÍGUO   ", name)
        for p in paths:
            print("             ", p)
        missing.append(name)

    else:
        print("NÃO ACHEI ", name)
        missing.append(name)

# ------------------------------------------------------------
# 2. Pastas com um modelo BERT salvo (config.json + pesos)
# ------------------------------------------------------------

print("\nPastas com modelo BERT encontradas:")

bert_candidates = []

for root, dirs, files in os.walk(SRC):
    if "config.json" in files and (
        "model.safetensors" in files or "pytorch_model.bin" in files
    ):
        bert_candidates.append(root)
        print(" -", root)

if not bert_candidates:
    print(" (nenhuma)")

# Depois de ver a lista acima, preencha com a pasta do BERT
# Head-Tail ORIGINAL (se não souber qual é, pergunte à outra pessoa;
# a célula final confere se é o modelo certo).

BERT_SRC = "/content/plnEp2/modelo_final/modelos/bert_headtail_final_model"  # exemplo: "/content/plnEp2/modelo_final/modelos/bert_headtail"

if BERT_SRC is not None:

    target = os.path.join(DST, "bert_headtail_final_model")

    if os.path.exists(target):
        shutil.rmtree(target)

    shutil.copytree(BERT_SRC, target)

    print("\nBERT copiado para:", target)

# ------------------------------------------------------------
# 3. Resumo
# ------------------------------------------------------------

print("\n" + "=" * 60)

if missing:
    print("Faltando ou ambíguos:", missing)
else:
    print("Todos os .npy necessários foram copiados.")

print("BERT final presente:",
      os.path.isdir(os.path.join(DST, "bert_headtail_final_model")))

OK         tfidf_word_oof.npy <- /content/plnEp2/modelo_final/probabilidades/tfidf_word_oof.npy
OK         tfidf_word_test1.npy <- /content/plnEp2/modelo_final/probabilidades/tfidf_word_test1.npy
OK         qwen06_lr_oof.npy <- /content/plnEp2/modelo_final/probabilidades/qwen06_lr_oof.npy
OK         qwen06_lr_test1.npy <- /content/plnEp2/modelo_final/probabilidades/qwen06_lr_test1.npy
OK         qwen4_lr_oof.npy <- /content/plnEp2/modelo_final/probabilidades/qwen4_lr_oof.npy
OK         qwen4_lr_test1.npy <- /content/plnEp2/modelo_final/probabilidades/qwen4_lr_test1.npy
OK         qwen06_train_embeddings.npy <- /content/plnEp2/modelo_final/embeddings/qwen06_train_embeddings.npy
OK         qwen06_test1_embeddings.npy <- /content/plnEp2/modelo_final/embeddings/qwen06_test1_embeddings.npy
OK         qwen4_train_embeddings.npy <- /content/plnEp2/modelo_final/embeddings/qwen4_train_embeddings.npy
OK         qwen4_test1_embeddings.npy <- /content/plnEp2/modelo_final/embeddings/qwen4_test1_emb

#Folds, avaliação e modelos TF-IDF

In [16]:
# @title
# ============================================================
# 1. CRIAÇÃO DOS FOLDS
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

FOLDS = list(
    skf.split(
        np.zeros(len(y_train_encoded)),
        y_train_encoded
    )
)

print("Folds criados:")

for i, (train_idx, val_idx) in enumerate(FOLDS, start=1):

    print(
        f"Fold {i}: "
        f"{len(train_idx)} treino | "
        f"{len(val_idx)} validação"
    )


# ============================================================
# 2. FUNÇÕES DE AVALIAÇÃO
# ============================================================

def evaluate_predictions(y_true, probabilities, model_name):

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    print("\n" + "=" * 65)
    print("RESULTADOS:", model_name)
    print("=" * 65)

    print(
        "Accuracy:",
        round(accuracy_score(y_true, predictions), 4)
    )

    print(
        "Macro-F1:",
        round(f1_score(
            y_true,
            predictions,
            average="macro"
        ), 4)
    )

    print(
        "Macro-Precision:",
        round(precision_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ), 4)
    )

    print(
        "Macro-Recall:",
        round(recall_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ), 4)
    )

    print("\nClassification Report:")

    print(
        classification_report(
            y_true,
            predictions,
            labels=[0, 1, 2],
            target_names=CLASSES,
            zero_division=0
        )
    )

    print("Confusion Matrix:")

    print(
        confusion_matrix(
            y_true,
            predictions,
            labels=[0, 1, 2]
        )
    )

    return predictions


# ============================================================
# 3. FUNÇÃO GERAL PARA TF-IDF
# ============================================================

def run_tfidf_model(name, vectorizer, C=1.0):

    print("\n" + "=" * 65)
    print("INICIANDO:", name)
    print("=" * 65)

    oof_path = f"{OUTPUT_DIR}/{name}_oof.npy"
    test_path = f"{OUTPUT_DIR}/{name}_test1.npy"

    # Reaproveitar resultados existentes
    if os.path.exists(oof_path) and os.path.exists(test_path):

        print("Resultados encontrados. Carregando...")

        return (
            np.load(oof_path),
            np.load(test_path)
        )

    pipeline = Pipeline([
        ("tfidf", vectorizer),

        ("lr", LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=2000,
            random_state=SEED
        ))
    ])

    # Previsões OOF
    oof_probabilities = cross_val_predict(
        pipeline,
        X_train_clean,
        y_train_encoded,
        cv=FOLDS,
        method="predict_proba",
        n_jobs=1
    )

    # Treinamento final com 100% dos dados
    pipeline.fit(
        X_train_clean,
        y_train_encoded
    )

    # Previsões do Teste 1
    test_probabilities = pipeline.predict_proba(
        X_test1_clean
    )

    # Garantir ordem das classes
    assert np.array_equal(
        pipeline.named_steps["lr"].classes_,
        np.array([0, 1, 2])
    )

    # Salvar resultados
    np.save(oof_path, oof_probabilities)
    np.save(test_path, test_probabilities)

    evaluate_predictions(
        y_test1_encoded,
        test_probabilities,
        name
    )

    del pipeline

    gc.collect()

    return oof_probabilities, test_probabilities


# ============================================================
# 4. MODELO 1 — TF-IDF WORD
# ============================================================

word_oof, word_test = run_tfidf_model(

    name="tfidf_word",

    vectorizer=TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 3),
        sublinear_tf=True,
        lowercase=True,
        use_idf=True,
        smooth_idf=True,
        norm="l2"
    ),

    C=1.0
)


# ============================================================
# 5. MODELO 2 — TF-IDF CHAR
# ============================================================

char_oof, char_test = run_tfidf_model(

    name="tfidf_char",

    vectorizer=TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        sublinear_tf=True,
        lowercase=True,
        use_idf=True,
        smooth_idf=True,
        norm="l2",
        min_df=2,
        max_features=300000
    ),

    C=1.0
)

print("\nModelos TF-IDF concluídos.")

Folds criados:
Fold 1: 13394 treino | 6698 validação
Fold 2: 13395 treino | 6697 validação
Fold 3: 13395 treino | 6697 validação

INICIANDO: tfidf_word
Resultados encontrados. Carregando...

INICIANDO: tfidf_char

RESULTADOS: tfidf_char
Accuracy: 0.4678
Macro-F1: 0.4661
Macro-Precision: 0.4652
Macro-Recall: 0.4678

Classification Report:
              precision    recall  f1-score   support

          c1       0.49      0.52      0.50       300
        c234       0.40      0.37      0.38       300
          c5       0.51      0.52      0.51       300

    accuracy                           0.47       900
   macro avg       0.47      0.47      0.47       900
weighted avg       0.47      0.47      0.47       900

Confusion Matrix:
[[156  87  57]
 [ 96 110  94]
 [ 67  78 155]]

Modelos TF-IDF concluídos.


#Qwen3 Embedding: extração dos embeddings e classificadores

In [17]:
# @title
# ============================================================
# 1. CONFIGURAÇÕES QWEN
# ============================================================

QWEN_INSTRUCTION = (
    "Represent the meaning of a Brazilian Portuguese public authority "
    "response for classification according to its clarity, completeness, "
    "and comprehensibility."
)

QWEN_MAX_LENGTH = 512

QWEN_06B = "Qwen/Qwen3-Embedding-0.6B"
QWEN_4B = "Qwen/Qwen3-Embedding-4B"


# ============================================================
# 2. POOLING DO ÚLTIMO TOKEN
# ============================================================

def last_token_pool(last_hidden_states):

    # Os tokens são preenchidos à esquerda.
    # Portanto, a última posição corresponde ao último token real.

    return last_hidden_states[:, -1, :]


# ============================================================
# 3. EXTRAÇÃO DOS EMBEDDINGS
# ============================================================

def extract_qwen_embeddings(
    model_name,
    texts,
    batch_size=4,
    max_length=QWEN_MAX_LENGTH
):

    print("\nCarregando:", model_name)

    use_amp = DEVICE.type == "cuda"

    tokenizer_qwen = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="left",
        trust_remote_code=True
    )

    # Garantir padding à esquerda
    tokenizer_qwen.padding_side = "left"

    model_qwen = AutoModel.from_pretrained(
        model_name,
        torch_dtype=(
            torch.float16 if use_amp else torch.float32
        ),
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )

    model_qwen = model_qwen.to(DEVICE)
    model_qwen.eval()

    formatted_texts = [
        f"Instruct: {QWEN_INSTRUCTION}\nQuery: {text}"
        for text in texts
    ]

    all_embeddings = []

    for start in range(0, len(formatted_texts), batch_size):

        batch_texts = formatted_texts[
            start:start + batch_size
        ]

        inputs = tokenizer_qwen(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        with torch.no_grad():

            if use_amp:

                with torch.cuda.amp.autocast():

                    outputs = model_qwen(**inputs)

                    embeddings = last_token_pool(
                        outputs.last_hidden_state
                    )

            else:

                outputs = model_qwen(**inputs)

                embeddings = last_token_pool(
                    outputs.last_hidden_state
                )

            # Normalização L2
            embeddings = F.normalize(
                embeddings.float(),
                p=2,
                dim=1
            )

        all_embeddings.append(
            embeddings.cpu().numpy()
        )

        processed = min(
            start + batch_size,
            len(formatted_texts)
        )

        if processed % (batch_size * 50) == 0 or processed == len(formatted_texts):

            print(
                f"Processados: {processed}/{len(formatted_texts)}"
            )

    embeddings = np.concatenate(
        all_embeddings,
        axis=0
    )

    del model_qwen
    del tokenizer_qwen
    del all_embeddings

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return embeddings


# ============================================================
# 4. CLASSIFICADOR LOGISTIC REGRESSION
# ============================================================

def run_embedding_classifier(
    name,
    train_embeddings,
    test_embeddings
):

    print("\n" + "=" * 65)
    print("CLASSIFICADOR:", name)
    print("=" * 65)

    oof_path = f"{OUTPUT_DIR}/{name}_oof.npy"
    test_path = f"{OUTPUT_DIR}/{name}_test1.npy"

    if os.path.exists(oof_path) and os.path.exists(test_path):

        print("Previsões encontradas. Carregando...")

        return (
            np.load(oof_path),
            np.load(test_path)
        )

    lr = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=2000,
        random_state=SEED
    )

    # OOF: cada exemplo é previsto por um classificador
    # que não foi treinado com aquele exemplo.
    oof_probabilities = cross_val_predict(
        lr,
        train_embeddings,
        y_train_encoded,
        cv=FOLDS,
        method="predict_proba",
        n_jobs=1
    )

    # Modelo final com todo o treinamento
    lr.fit(
        train_embeddings,
        y_train_encoded
    )

    test_probabilities = lr.predict_proba(
        test_embeddings
    )

    assert np.array_equal(
        lr.classes_,
        np.array([0, 1, 2])
    )

    np.save(oof_path, oof_probabilities)
    np.save(test_path, test_probabilities)

    evaluate_predictions(
        y_test1_encoded,
        test_probabilities,
        name
    )

    del lr
    gc.collect()

    return (
        oof_probabilities,
        test_probabilities
    )


# ============================================================
# 5. QWEN 0.6B
# ============================================================

qwen06_train_path = (
    f"{OUTPUT_DIR}/qwen06_train_embeddings.npy"
)

qwen06_test_path = (
    f"{OUTPUT_DIR}/qwen06_test1_embeddings.npy"
)

if (
    os.path.exists(qwen06_train_path)
    and os.path.exists(qwen06_test_path)
):

    print("Embeddings Qwen 0.6B encontrados.")

    qwen06_train_embeddings = np.load(
        qwen06_train_path
    )

    qwen06_test_embeddings = np.load(
        qwen06_test_path
    )

else:

    qwen06_train_embeddings = extract_qwen_embeddings(
        QWEN_06B,
        X_train_raw,
        batch_size=4
    )

    np.save(
        qwen06_train_path,
        qwen06_train_embeddings
    )

    qwen06_test_embeddings = extract_qwen_embeddings(
        QWEN_06B,
        X_test1_raw,
        batch_size=4
    )

    np.save(
        qwen06_test_path,
        qwen06_test_embeddings
    )


print(
    "Qwen 0.6B — treinamento:",
    qwen06_train_embeddings.shape
)

print(
    "Qwen 0.6B — Teste 1:",
    qwen06_test_embeddings.shape
)


# Classificador
qwen06_oof, qwen06_test = run_embedding_classifier(
    "qwen06_lr",
    qwen06_train_embeddings,
    qwen06_test_embeddings
)


# ============================================================
# 6. QWEN 4B — OPCIONAL
# ============================================================

qwen4_oof = None
qwen4_test = None

qwen4_train_embeddings = None
qwen4_test_embeddings = None

qwen4_train_path = (
    f"{OUTPUT_DIR}/qwen4_train_embeddings.npy"
)

qwen4_test_path = (
    f"{OUTPUT_DIR}/qwen4_test1_embeddings.npy"
)

# Verificar memória livre
if torch.cuda.is_available():

    free_memory_gb = (
        torch.cuda.mem_get_info()[0] / 1024**3
    )

    print(
        f"\nMemória livre antes do Qwen 4B: "
        f"{free_memory_gb:.2f} GB"
    )

else:

    free_memory_gb = 0


# Critério conservador de execução
RUN_QWEN_4B = (
    torch.cuda.is_available()
    and free_memory_gb >= 12
)

if RUN_QWEN_4B:

    try:

        if (
            os.path.exists(qwen4_train_path)
            and os.path.exists(qwen4_test_path)
        ):

            print("Embeddings Qwen 4B encontrados.")

            qwen4_train_embeddings = np.load(
                qwen4_train_path
            )

            qwen4_test_embeddings = np.load(
                qwen4_test_path
            )

        else:

            qwen4_train_embeddings = extract_qwen_embeddings(
                QWEN_4B,
                X_train_raw,
                batch_size=1
            )

            np.save(
                qwen4_train_path,
                qwen4_train_embeddings
            )

            qwen4_test_embeddings = extract_qwen_embeddings(
                QWEN_4B,
                X_test1_raw,
                batch_size=1
            )

            np.save(
                qwen4_test_path,
                qwen4_test_embeddings
            )

        qwen4_oof, qwen4_test = run_embedding_classifier(
            "qwen4_lr",
            qwen4_train_embeddings,
            qwen4_test_embeddings
        )

    except torch.cuda.OutOfMemoryError:

        print(
            "\nQwen 4B interrompido por falta de memória GPU."
        )

        qwen4_oof = None
        qwen4_test = None

        qwen4_train_embeddings = None
        qwen4_test_embeddings = None

        gc.collect()
        torch.cuda.empty_cache()

else:

    print(
        "\nQwen 4B não será executado nesta sessão. "
        "Memória GPU insuficiente para a tentativa conservadora."
    )


print("\nEtapa Qwen concluída.")

Embeddings Qwen 0.6B encontrados.
Qwen 0.6B — treinamento: (20092, 1024)
Qwen 0.6B — Teste 1: (900, 1024)

CLASSIFICADOR: qwen06_lr
Previsões encontradas. Carregando...

Memória livre antes do Qwen 4B: 14.46 GB
Embeddings Qwen 4B encontrados.

CLASSIFICADOR: qwen4_lr
Previsões encontradas. Carregando...

Etapa Qwen concluída.


# Geração das predições para o Teste 2

In [18]:
# ============================================================
# ROTULAÇÃO DO TESTE 2 — STACKING FINAL
# Combinação: TF-IDF Word + Qwen 0.6B + Qwen 4B + BERT Head-Tail ORIGINAL
#
# Pré-requisitos (mesma sessão do Colab, nesta ordem): células 3, 6, 8 e 12.
# Precisa existir em /content/ensemble_outputs:
#   - bert_headtail_oof.npy
#   - bert_headtail_test1.npy
#   - bert_headtail_final_model/   (BERT treinado com 100% do treino)
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import torch

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ------------------------------------------------------------
# 1. CAMINHOS E PRÉ-REQUISITOS
# ------------------------------------------------------------

TEST2_PATH = "/content/plnEp2/data/test2.xlsx"
TEST2_CLEAN_PATH = "/content/plnEp2/data/test2_limpo.csv"

OUTPUT_DIR = "/content/ensemble_outputs"
STACK_DIR = os.path.join(OUTPUT_DIR, "stacking_headtail_original")
MODEL_DIR = os.path.join(STACK_DIR, "modelos")
FINAL_EXPORT_DIR = "/content/plnEp2/modelo_final"

BERT_FINAL_DIR = os.path.join(OUTPUT_DIR, "bert_headtail_final_model")
BERT_OOF_PATH = os.path.join(OUTPUT_DIR, "bert_headtail_oof.npy")
BERT_TEST1_PATH = os.path.join(OUTPUT_DIR, "bert_headtail_test1.npy")

N_TEST2_EXPECTED = 897

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FINAL_EXPORT_DIR, exist_ok=True)

required_variables = [
    "X_train_clean",
    "X_train_raw",
    "X_test1_raw",
    "y_train_encoded",
    "qwen06_train_embeddings",
    "qwen4_train_embeddings",
    "extract_qwen_embeddings",
    "QWEN_06B",
    "QWEN_4B",
    "SEED",
    "DEVICE",
    "ID_TO_LABEL"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Variáveis ausentes: "
        + ", ".join(missing_variables)
        + ". Execute as células 3, 6, 8 e 12, nesta ordem."
    )

for path in [TEST2_PATH, TEST2_CLEAN_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

for path in [BERT_OOF_PATH, BERT_TEST1_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Arquivo do BERT Head-Tail original não encontrado: {path}"
        )

if not os.path.isdir(BERT_FINAL_DIR):
    raise FileNotFoundError(
        f"Modelo final do BERT Head-Tail não encontrado: {BERT_FINAL_DIR}"
    )

# ------------------------------------------------------------
# 2. CARREGAR TESTE 2 E VALIDAR ALINHAMENTO
# ------------------------------------------------------------

test2_df = pd.read_excel(TEST2_PATH)
test2_clean_df = pd.read_csv(TEST2_CLEAN_PATH)

for name, df in [
    ("Teste 2 original", test2_df),
    ("Teste 2 limpo", test2_clean_df)
]:
    if "resp_text" not in df.columns:
        raise ValueError(
            f"A coluna 'resp_text' não foi encontrada em {name}."
        )

if len(test2_df) != len(test2_clean_df):
    raise ValueError(
        "O Teste 2 original e o limpo possuem quantidades "
        "diferentes de linhas. Verifique o alinhamento."
    )

assert len(test2_df) == N_TEST2_EXPECTED, (
    f"Esperadas {N_TEST2_EXPECTED} instâncias no Teste 2, "
    f"encontradas {len(test2_df)}."
)

X_test2_raw = (
    test2_df["resp_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .tolist()
)

X_test2_clean = (
    test2_clean_df["resp_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .tolist()
)

print(f"Teste 2 carregado: {len(X_test2_raw)} exemplos.")

# ------------------------------------------------------------
# 3. PROBABILIDADES OOF E META-CLASSIFICADOR
#    Ordem das features: TF-IDF Word, Qwen 0.6B, Qwen 4B, BERT Head-Tail
# ------------------------------------------------------------

oof_paths = [
    os.path.join(OUTPUT_DIR, "tfidf_word_oof.npy"),
    os.path.join(OUTPUT_DIR, "qwen06_lr_oof.npy"),
    os.path.join(OUTPUT_DIR, "qwen4_lr_oof.npy"),
    BERT_OOF_PATH
]

for path in oof_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Probabilidades OOF não encontradas: {path}. "
            "Execute as células correspondentes antes de continuar."
        )

oof_components = [np.load(path) for path in oof_paths]

expected_train_shape = (len(y_train_encoded), 3)

for name, probabilities in zip(
    ["TF-IDF Word", "Qwen 0.6B", "Qwen 4B", "BERT Head-Tail"],
    oof_components
):
    assert probabilities.shape == expected_train_shape, (
        f"Dimensão OOF incorreta em {name}: {probabilities.shape}"
    )
    assert np.isfinite(probabilities).all(), (
        f"Valores inválidos nas probabilidades OOF de {name}."
    )

X_meta_train = np.hstack(oof_components)

assert X_meta_train.shape == (len(y_train_encoded), 12)

meta_model = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=2000
)

meta_model.fit(X_meta_train, y_train_encoded)

assert np.array_equal(meta_model.classes_, np.array([0, 1, 2]))

joblib.dump(
    meta_model,
    os.path.join(MODEL_DIR, "stacking_meta_model.pkl")
)

print("Meta-classificador treinado com as probabilidades OOF.")

# ------------------------------------------------------------
# 4. COMPONENTE 1 — TF-IDF WORD + REGRESSÃO LOGÍSTICA
# ------------------------------------------------------------

print("\n[1/4] Treinando TF-IDF Word...")

pipeline_word = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            sublinear_tf=True,
            lowercase=True,
            use_idf=True,
            smooth_idf=True,
            norm="l2"
        )
    ),
    (
        "lr",
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            random_state=SEED
        )
    )
])

pipeline_word.fit(X_train_clean, y_train_encoded)

word_test2_probs = pipeline_word.predict_proba(X_test2_clean)

assert np.array_equal(
    pipeline_word.named_steps["lr"].classes_,
    np.array([0, 1, 2])
)

joblib.dump(
    pipeline_word,
    os.path.join(MODEL_DIR, "tfidf_word_pipeline.pkl")
)

# ------------------------------------------------------------
# 5. COMPONENTE 2 — QWEN 0.6B + REGRESSÃO LOGÍSTICA
# ------------------------------------------------------------

print("\n[2/4] Extraindo embeddings do Qwen 0.6B...")

qwen06_test2_embeddings = extract_qwen_embeddings(
    QWEN_06B,
    X_test2_raw,
    batch_size=4
)

lr_qwen06 = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    random_state=SEED
)

lr_qwen06.fit(qwen06_train_embeddings, y_train_encoded)

qwen06_test2_probs = lr_qwen06.predict_proba(qwen06_test2_embeddings)

assert np.array_equal(lr_qwen06.classes_, np.array([0, 1, 2]))

joblib.dump(
    lr_qwen06,
    os.path.join(MODEL_DIR, "qwen06_logistic_regression.pkl")
)

# ------------------------------------------------------------
# 6. COMPONENTE 3 — QWEN 4B + REGRESSÃO LOGÍSTICA
# ------------------------------------------------------------

print("\n[3/4] Extraindo embeddings do Qwen 4B...")

qwen4_test2_embeddings = extract_qwen_embeddings(
    QWEN_4B,
    X_test2_raw,
    batch_size=1
)

lr_qwen4 = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    random_state=SEED
)

lr_qwen4.fit(qwen4_train_embeddings, y_train_encoded)

qwen4_test2_probs = lr_qwen4.predict_proba(qwen4_test2_embeddings)

assert np.array_equal(lr_qwen4.classes_, np.array([0, 1, 2]))

joblib.dump(
    lr_qwen4,
    os.path.join(MODEL_DIR, "qwen4_logistic_regression.pkl")
)

# ------------------------------------------------------------
# 7. COMPONENTE 4 — BERT HEAD-TAIL ORIGINAL
#    Modelo treinado com 100% do treino (salvo pela célula 10).
# ------------------------------------------------------------

print("\n[4/4] Gerando probabilidades do BERT Head-Tail original...")

bert_tok = AutoTokenizer.from_pretrained(BERT_FINAL_DIR)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_FINAL_DIR
).to(DEVICE)

bert_model.eval()


def encode_head_tail(text):

    ids = bert_tok.encode(text, add_special_tokens=False)

    # 127 tokens iniciais + 127 finais (mesma regra da célula 10)
    if len(ids) > 254:
        ids = ids[:127] + ids[-127:]

    return bert_tok.build_inputs_with_special_tokens(ids)


@torch.no_grad()
def predict_headtail(texts, batch_size=32):

    ids_list = [encode_head_tail(t) for t in texts]

    outputs = []

    for start in range(0, len(ids_list), batch_size):

        batch = ids_list[start:start + batch_size]
        max_len = max(len(x) for x in batch)

        input_ids = torch.tensor([
            x + [bert_tok.pad_token_id] * (max_len - len(x))
            for x in batch
        ]).to(DEVICE)

        attention_mask = torch.tensor([
            [1] * len(x) + [0] * (max_len - len(x))
            for x in batch
        ]).to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits

        outputs.append(
            torch.softmax(logits.float(), dim=1).cpu().numpy()
        )

    return np.concatenate(outputs, axis=0)


# Checagem: o modelo carregado reproduz as previsões do Teste 1 salvas?
bert_test1_check = predict_headtail(X_test1_raw)
bert_test1_saved = np.load(BERT_TEST1_PATH)

agreement = float(
    (bert_test1_check.argmax(1) == bert_test1_saved.argmax(1)).mean()
)

print(f"Concordância com as previsões salvas do Teste 1: {agreement:.4f}")

assert agreement >= 0.97, (
    "O BERT carregado não reproduz as previsões salvas do Teste 1. "
    "Verifique se bert_headtail_final_model é o modelo correto."
)

bert_test2_probs = predict_headtail(X_test2_raw)

assert bert_test2_probs.shape == (len(test2_df), 3)

# ------------------------------------------------------------
# 8. STACKING — MESMA ORDEM DO TREINO DO META-CLASSIFICADOR
# ------------------------------------------------------------

X_meta_test2 = np.hstack([
    word_test2_probs,
    qwen06_test2_probs,
    qwen4_test2_probs,
    bert_test2_probs
])

assert X_meta_test2.shape == (len(test2_df), 12)
assert np.isfinite(X_meta_test2).all()

stacking_test2_probabilities = meta_model.predict_proba(X_meta_test2)
stacking_test2_predictions = meta_model.predict(X_meta_test2)

final_predictions_labels = [
    ID_TO_LABEL[int(idx)]
    for idx in stacking_test2_predictions
]

# ------------------------------------------------------------
# 9. EXPORTAR RESULTADOS
# ------------------------------------------------------------

test2_results_df = test2_df.copy()

test2_results_df["clarity"] = final_predictions_labels
test2_results_df["predicted_clarity"] = final_predictions_labels
test2_results_df["prob_c1"] = stacking_test2_probabilities[:, 0]
test2_results_df["prob_c234"] = stacking_test2_probabilities[:, 1]
test2_results_df["prob_c5"] = stacking_test2_probabilities[:, 2]

output_path = os.path.join(STACK_DIR, "stacking_test2_predictions.csv")
export_path = os.path.join(FINAL_EXPORT_DIR, "stacking_test2_predictions.csv")

test2_results_df.to_csv(output_path, index=False, encoding="utf-8-sig")
test2_results_df.to_csv(export_path, index=False, encoding="utf-8-sig")

np.save(
    os.path.join(STACK_DIR, "stacking_test2_probabilities.npy"),
    stacking_test2_probabilities
)

metadata = {
    "architecture": "Stacking",
    "base_models": [
        "TF-IDF Word + Logistic Regression",
        "Qwen 0.6B + Logistic Regression",
        "Qwen 4B + Logistic Regression",
        "BERTimbau Base Head-Tail original"
    ],
    "meta_classifier": "LogisticRegression",
    "meta_classifier_parameters": {
        "C": 1,
        "class_weight": "balanced",
        "max_iter": 2000
    },
    "feature_order": [
        "TF-IDF Word",
        "Qwen 0.6B",
        "Qwen 4B",
        "BERT Head-Tail original"
    ],
    "number_of_meta_features": 12,
    "bert_parameters": {
        "learning_rate": 3e-5,
        "weight_decay": 0.01,
        "epochs": 3,
        "max_length": 256,
        "head_tokens": 127,
        "tail_tokens": 127,
        "seed": int(SEED)
    },
    "classes": ["c1", "c234", "c5"],
    "test2_examples": len(test2_df)
}

with open(
    os.path.join(MODEL_DIR, "stacking_test2_metadata.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)

print("\n" + "=" * 65)
print("ROTULAÇÃO DO TESTE 2 CONCLUÍDA")
print("=" * 65)
print("Exemplos rotulados:", len(test2_results_df))
print("Arquivo:", output_path)
print("Cópia:", export_path)

print("\nDistribuição dos rótulos previstos:")
print(pd.Series(final_predictions_labels).value_counts())

display(
    test2_results_df[
        ["resp_text", "clarity", "prob_c1", "prob_c234", "prob_c5"]
    ].head(10)
)

Teste 2 carregado: 897 exemplos.
Meta-classificador treinado com as probabilidades OOF.

[1/4] Treinando TF-IDF Word...

[2/4] Extraindo embeddings do Qwen 0.6B...

Carregando: Qwen/Qwen3-Embedding-0.6B


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

... (11 outputs left out)

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Processados: 50/897
Processados: 100/897
Processados: 150/897
Processados: 200/897
Processados: 250/897
Processados: 300/897
Processados: 350/897
Processados: 400/897
Processados: 450/897
Processados: 500/897
Processados: 550/897
Processados: 600/897
Processados: 650/897
Processados: 700/897
Processados: 750/897
Processados: 800/897
Processados: 850/897
Processados: 897/897

[4/4] Gerando probabilidades do BERT Head-Tail original...
Concordância com as previsões salvas do Teste 1: 1.0000

ROTULAÇÃO DO TESTE 2 CONCLUÍDA
Exemplos rotulados: 897
Arquivo: /content/ensemble_outputs/stacking_headtail_original/stacking_test2_predictions.csv
Cópia: /content/plnEp2/modelo_final/stacking_test2_predictions.csv

Distribuição dos rótulos previstos:
c1      337
c5      328
c234    232
Name: count, dtype: int64


,resp_text,clarity,prob_c1,prob_c234,prob_c5
0,A Universidade Federal de Goiás (UFG) não tem...,c1,0.666846,0.207814,0.125340
1,"Prezado Sr. Álvaro, Em atendimento à solici...",c5,0.164422,0.320911,0.514667
2,"Prezado Senhor, 1. Em atenção ao pedido de...",c1,0.548202,0.325566,0.126232
3,Respondente: Serviço de Informação ao Cidadão...,c234,0.355904,0.407328,0.236769
4,Bom dia! Sua defesa aguarda julgamento perant...,c1,0.340362,0.339995,0.319644
5,"Prezado(a) Senhor(a), Com base nas informa...",c234,0.367128,0.404399,0.228473
6,Prezado Senhor. Em atenção ao seu pe...,c5,0.248818,0.271020,0.480162
7,A Assistência da Pró-reitoria de Gestão de Pe...,c5,0.141790,0.322395,0.535816
8,"Prezado Senhor, Esclarecemos que conforme pre...",c1,0.546534,0.342229,0.111237
9,"Senhor Francisco, O Serviço de Informações a...",c1,0.462834,0.392452,0.144714


In [19]:
import pandas as pd

df = pd.read_csv("/content/plnEp2/modelo_final/stacking_test2_predictions.csv")

out = df[["resp_text", "clarity"]]

assert len(out) == 897
assert out["clarity"].isin(["c1", "c234", "c5"]).all()

out.to_excel("/content/plnEp2/modelo_final/test2_rotulado.xlsx", index=False)

print(out["clarity"].value_counts())
out.head()

clarity
c1      337
c5      328
c234    232
Name: count, dtype: int64


,resp_text,clarity
0,A Universidade Federal de Goiás (UFG) não tem...,c1
1,"Prezado Sr. Álvaro, Em atendimento à solici...",c5
2,"Prezado Senhor, 1. Em atenção ao pedido de...",c1
3,Respondente: Serviço de Informação ao Cidadão...,c234
4,Bom dia! Sua defesa aguarda julgamento perant...,c1
